# 06 — High-Cost Classification

**Objective** — identify high-cost hospitalisation events using a threshold learned from training charges only.

**Input** — cleaned analytical dataset and training-only preprocessing.

**Output** — selected classifier, training-derived threshold, and final test metrics.

## 1. Prepare the Grouped Split and Risk Labels

In [1]:
from pathlib import Path
import sys
import pandas as pd

current_directory = Path.cwd().resolve()
project_root = next(
    directory for directory in [current_directory, *current_directory.parents]
    if (directory / 'src').is_dir() and (directory / 'notebooks').is_dir()
)
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from src.features import prepare_feature_frame, build_preprocessor
from src.modeling import grouped_train_test_split, derive_high_cost_labels, get_classification_candidates, compare_models_cv
from src.evaluation import classification_metrics

processed_data_path = project_root / 'data' / 'processed' / 'analytical_dataset.csv'
analytical_dataset = pd.read_csv(processed_data_path)
features, target_charges, customer_groups = prepare_feature_frame(analytical_dataset)
training_features, test_features, training_charges, test_charges, training_groups, test_groups = grouped_train_test_split(
    features, target_charges, customer_groups
)
training_high_cost_labels, test_high_cost_labels, high_cost_threshold = derive_high_cost_labels(
    training_charges, test_charges
)
preprocessor = build_preprocessor(training_features)

## 2. Compare Candidate Models

In [2]:
classification_models = get_classification_candidates(preprocessor)
classification_comparison = compare_models_cv(
    classification_models, training_features, training_high_cost_labels, training_groups, 'classification'
)
classification_comparison

,model,cv_mean,cv_std
0,HistGradientBoosting,0.940314,0.018394
1,Random Forest,0.927139,0.024634
2,Logistic Regression,0.837324,0.036460
3,Dummy Prior,0.100267,0.013764


## 3. Evaluate the Selected Model

In [3]:
selected_model_name = classification_comparison.iloc[0]['model']
selected_classifier = classification_models[selected_model_name].fit(
    training_features, training_high_cost_labels
)
classification_probability_scores = selected_classifier.predict_proba(test_features)[:, 1]
classification_predictions = (classification_probability_scores >= 0.5).astype(int)
{
    'high_cost_threshold': high_cost_threshold,
    'selected_model': selected_model_name,
    **classification_metrics(
        test_high_cost_labels, classification_probability_scores, classification_predictions
    ),
}

{'high_cost_threshold': 34962.096000000005,
 'selected_model': 'HistGradientBoosting',
 'precision': 0.9024390243902439,
 'recall': 0.8604651162790697,
 'f1': 0.8809523809523809,
 'roc_auc': 0.9925581395348837,
 'pr_auc': 0.9684662077536874}

## Takeaways

PR-AUC is the main selection metric because high-cost cases are rare. Recall also matters because false negatives are high-cost cases the model fails to flag.